# Genetic Mapping Extension: Interference and Coefficient of Coincidence

**Advanced Pattern Hunting: When Crossovers Aren't Truly Independent**

## The Biological Reality

In our previous notebook, we assumed crossovers are **independent** events (Poisson assumption). But real chromosomes show **interference**: one crossover makes nearby crossovers less likely!

This module covers:
1. What is interference and why does it occur?
2. Coefficient of Coincidence (COC)
3. Interference calculation and interpretation
4. Correcting map distances for interference
5. Real-world examples from fish and earthworm genetics

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import poisson
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## Part 1: Understanding Interference

### The Basic Concept

**Interference**: The phenomenon where a crossover in one region **reduces** the probability of another crossover occurring nearby.

Think of it like this:
- When one crossover happens, the chromosome structure temporarily "stabilizes"
- Nearby regions become less likely to cross over
- This creates a "zone of interference" around each crossover

### Why Does This Matter?

**Without interference** (pure Poisson):
```
A -------- B -------- C
Double crossovers occur at expected frequency
```

**With interference** (biological reality):
```
A -------- B -------- C
Double crossovers occur LESS than expected
→ More accurate genetic maps!
```

## Part 2: Three-Point Test Cross Analysis

### The Eight Gamete Classes

In a three-point cross (AaBbCc × aabbcc), we get 8 possible gamete types:

| Gamete Type | Crossover Pattern | Category |
|-------------|-------------------|----------|
| ABC | No crossovers | Parental |
| abc | No crossovers | Parental |
| AbC | Single (region I) | SCO I |
| aBc | Single (region I) | SCO I |
| ABc | Single (region II) | SCO II |
| abC | Single (region II) | SCO II |
| Abc | Double (both regions) | DCO |
| aBC | Double (both regions) | DCO |

**Key**: DCO (Double Crossover) class is crucial for measuring interference!

In [ ]:
def visualize_crossover_classes():
    """
    Visualize the eight gamete classes from three-point cross
    """
    fig, axes = plt.subplots(4, 2, figsize=(14, 16))
    axes = axes.flatten()
    
    # Define the eight classes
    classes = [
        ('ABC', 'No crossover', 'Parental', 'lightblue', []),
        ('abc', 'No crossover', 'Parental', 'lightblue', []),
        ('AbC', 'Single CO (Region I)', 'SCO I', 'lightcoral', [0.3]),
        ('aBc', 'Single CO (Region I)', 'SCO I', 'lightcoral', [0.3]),
        ('ABc', 'Single CO (Region II)', 'SCO II', 'lightgreen', [0.7]),
        ('abC', 'Single CO (Region II)', 'SCO II', 'lightgreen', [0.7]),
        ('Abc', 'Double CO (Both)', 'DCO', 'gold', [0.3, 0.7]),
        ('aBC', 'Double CO (Both)', 'DCO', 'gold', [0.3, 0.7])
    ]
    
    for idx, (genotype, description, category, color, crossover_pos) in enumerate(classes):
        ax = axes[idx]
        
        # Draw chromosome
        ax.plot([0, 1], [0.5, 0.5], 'k-', linewidth=8, solid_capstyle='round')
        
        # Draw gene positions
        gene_positions = [0.2, 0.5, 0.8]
        gene_names = ['A', 'B', 'C']
        
        for pos, gene in zip(gene_positions, gene_names):
            ax.plot([pos, pos], [0.35, 0.65], 'b-', linewidth=4)
            ax.text(pos, 0.75, gene, ha='center', fontsize=14, fontweight='bold')
        
        # Draw crossover positions
        for co_pos in crossover_pos:
            # X mark for crossover
            ax.plot([co_pos-0.03, co_pos+0.03], [0.4, 0.6], 'r-', linewidth=4)
            ax.plot([co_pos-0.03, co_pos+0.03], [0.6, 0.4], 'r-', linewidth=4)
            ax.plot(co_pos, 0.5, 'ro', markersize=12)
        
        # Labels
        ax.text(0.5, 0.15, f"Gamete: {genotype}", ha='center', fontsize=13, 
               fontweight='bold', bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))
        ax.text(0.5, 0.05, description, ha='center', fontsize=11)
        
        ax.set_xlim(-0.1, 1.1)
        ax.set_ylim(0, 0.9)
        ax.axis('off')
        ax.set_title(f"{category}", fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/eight_gamete_classes.png', dpi=300, bbox_inches='tight')
    plt.show()

visualize_crossover_classes()
print("\n✓ Visual guide to eight gamete classes created!")

## Part 3: Coefficient of Coincidence (COC)

### Definition

**COC measures how often double crossovers actually occur compared to what we'd expect if crossovers were independent.**

### Formula

$$\text{COC} = \frac{\text{Observed DCO}}{\text{Expected DCO}}$$

Where:
- **Observed DCO** = actual count of double crossovers from data
- **Expected DCO** = (RF in region I) × (RF in region II) × (total offspring)

### Interpretation

- **COC = 1.0**: No interference (crossovers are independent)
- **COC < 1.0**: Positive interference (DCOs less than expected) ← **Most common**
- **COC > 1.0**: Negative interference (DCOs more than expected) ← **Very rare**
- **COC = 0**: Complete interference (no double crossovers at all)

## Part 4: Interference Calculation

### Formula

$$\text{Interference (I)} = 1 - \text{COC}$$

### Interpretation

- **I = 0**: No interference
- **I = 0.5**: Moderate interference (50% reduction in DCOs)
- **I = 1.0**: Complete interference (100% reduction, no DCOs)
- **I < 0**: Negative interference (more DCOs than expected) ← rare

### Typical Values in Different Organisms

| Organism | Typical Interference | Notes |
|----------|---------------------|-------|
| *Drosophila* | 0.5 - 0.7 | Strong interference |
| Maize | 0.3 - 0.5 | Moderate interference |
| Humans | 0.4 - 0.6 | Moderate-strong |
| Yeast | 0.1 - 0.3 | Weak interference |
| Fish (teleosts) | 0.2 - 0.5 | Variable by species |

In [ ]:
def calculate_coc_and_interference(observed_dco, rf_region1, rf_region2, total_offspring):
    """
    Calculate coefficient of coincidence and interference
    
    Parameters:
    observed_dco: number of double crossover offspring observed
    rf_region1: recombination frequency in region I (as decimal)
    rf_region2: recombination frequency in region II (as decimal)
    total_offspring: total number of offspring
    
    Returns:
    Dictionary with COC, interference, and related statistics
    """
    # Expected DCO if independent
    expected_dco = rf_region1 * rf_region2 * total_offspring
    
    # Coefficient of Coincidence
    coc = observed_dco / expected_dco if expected_dco > 0 else 0
    
    # Interference
    interference = 1 - coc
    
    return {
        'observed_dco': observed_dco,
        'expected_dco': expected_dco,
        'coc': coc,
        'interference': interference,
        'interference_percent': interference * 100,
        'reduction_in_dco': expected_dco - observed_dco
    }

# Example calculation
print("="*70)
print("EXAMPLE: Interference Calculation")
print("="*70)
print("\nThree-point testcross data:")
print("  Region I (A-B): 18% recombination")
print("  Region II (B-C): 12% recombination")
print("  Total offspring: 1000")
print("  Observed double crossovers: 12")

result = calculate_coc_and_interference(
    observed_dco=12,
    rf_region1=0.18,
    rf_region2=0.12,
    total_offspring=1000
)

print("\nCalculations:")
print(f"  Expected DCO (if independent) = 0.18 × 0.12 × 1000 = {result['expected_dco']:.1f}")
print(f"  Observed DCO = {result['observed_dco']}")
print(f"\n  COC = {result['observed_dco']}/{result['expected_dco']:.1f} = {result['coc']:.3f}")
print(f"  Interference = 1 - {result['coc']:.3f} = {result['interference']:.3f}")

print("\n" + "="*70)
print("INTERPRETATION")
print("="*70)
print(f"\n✓ Coefficient of Coincidence: {result['coc']:.3f}")
print(f"✓ Interference: {result['interference']:.3f} ({result['interference_percent']:.1f}%)")
print(f"\nMeaning: One crossover reduces the chance of a second nearby")
print(f"crossover by {result['interference_percent']:.1f}%. This is MODERATE interference.")
print(f"\n{result['reduction_in_dco']:.1f} fewer DCOs than expected due to interference.")
print("="*70)

## Part 5: Complete Three-Point Cross Analysis

### Step-by-Step Workflow

In [ ]:
class ThreePointCross:
    """
    Complete analysis of three-point cross data including interference
    """
    
    def __init__(self, data_dict):
        """
        data_dict should contain counts for each gamete class
        Keys: 'ABC', 'abc', 'AbC', 'aBc', 'ABc', 'abC', 'Abc', 'aBC'
        """
        self.data = data_dict
        self.total = sum(data_dict.values())
        self.analyze()
    
    def identify_gene_order(self):
        """Identify middle gene by finding DCO class"""
        # Find the rarest class (DCO)
        min_count = min(self.data.values())
        dco_classes = [k for k, v in self.data.items() if v == min_count]
        
        # DCO classes have the middle gene switched
        # Logic to determine gene order from DCO pattern
        # (simplified for this example)
        return 'A-B-C'  # Placeholder
    
    def calculate_map_distances(self):
        """Calculate map distances from SCO and DCO classes"""
        # Region I: SCO I + DCO
        sco1 = self.data.get('AbC', 0) + self.data.get('aBc', 0)
        dco = self.data.get('Abc', 0) + self.data.get('aBC', 0)
        
        region1_recombinants = sco1 + dco
        rf_region1 = region1_recombinants / self.total
        
        # Region II: SCO II + DCO
        sco2 = self.data.get('ABc', 0) + self.data.get('abC', 0)
        region2_recombinants = sco2 + dco
        rf_region2 = region2_recombinants / self.total
        
        return {
            'region1_cM': rf_region1 * 100,
            'region2_cM': rf_region2 * 100,
            'total_cM': (rf_region1 + rf_region2) * 100,
            'rf_region1': rf_region1,
            'rf_region2': rf_region2,
            'dco_count': dco
        }
    
    def calculate_interference(self, distances):
        """Calculate COC and interference"""
        expected_dco = distances['rf_region1'] * distances['rf_region2'] * self.total
        observed_dco = distances['dco_count']
        
        coc = observed_dco / expected_dco if expected_dco > 0 else 0
        interference = 1 - coc
        
        return {
            'expected_dco': expected_dco,
            'observed_dco': observed_dco,
            'coc': coc,
            'interference': interference
        }
    
    def analyze(self):
        """Complete analysis"""
        self.gene_order = self.identify_gene_order()
        self.distances = self.calculate_map_distances()
        self.interference_data = self.calculate_interference(self.distances)
    
    def report(self):
        """Generate comprehensive report"""
        print("\n" + "="*70)
        print("THREE-POINT CROSS ANALYSIS REPORT")
        print("="*70)
        
        print("\n1. DATA SUMMARY")
        print("-" * 70)
        for genotype, count in sorted(self.data.items()):
            percent = (count / self.total) * 100
            print(f"   {genotype:5s}: {count:4d} ({percent:5.2f}%)")
        print(f"   Total: {self.total:4d}")
        
        print("\n2. GENE ORDER")
        print("-" * 70)
        print(f"   {self.gene_order}")
        
        print("\n3. MAP DISTANCES")
        print("-" * 70)
        print(f"   Region I:  {self.distances['region1_cM']:.2f} cM")
        print(f"   Region II: {self.distances['region2_cM']:.2f} cM")
        print(f"   Total map: {self.distances['total_cM']:.2f} cM")
        
        print("\n4. INTERFERENCE ANALYSIS")
        print("-" * 70)
        print(f"   Expected DCO:  {self.interference_data['expected_dco']:.2f}")
        print(f"   Observed DCO:  {self.interference_data['observed_dco']:.0f}")
        print(f"   COC:           {self.interference_data['coc']:.3f}")
        print(f"   Interference:  {self.interference_data['interference']:.3f} ({self.interference_data['interference']*100:.1f}%)")
        
        print("\n5. BIOLOGICAL INTERPRETATION")
        print("-" * 70)
        
        i = self.interference_data['interference']
        if i < 0:
            print("   ⚠️  NEGATIVE interference: More DCOs than expected (unusual!)")
        elif i == 0:
            print("   ✓ NO interference: Crossovers are independent")
        elif i < 0.3:
            print("   ✓ WEAK interference: Crossovers slightly dependent")
        elif i < 0.6:
            print("   ✓ MODERATE interference: Typical for most organisms")
        else:
            print("   ✓ STRONG interference: One crossover strongly inhibits nearby")
        
        reduction = self.interference_data['expected_dco'] - self.interference_data['observed_dco']
        print(f"\n   {reduction:.1f} fewer DCOs observed than expected due to interference.")
        print("\n" + "="*70)

# Example: Labeo rohita microsatellite data
print("\n\n" + "#"*70)
print("# EXAMPLE 1: Labeo rohita (Indian Major Carp) Genetic Analysis")
print("#"*70)

labeo_data = {
    'ABC': 310,  # Parental
    'abc': 305,  # Parental
    'AbC': 85,   # SCO region I
    'aBc': 88,   # SCO region I
    'ABc': 62,   # SCO region II
    'abC': 58,   # SCO region II
    'Abc': 8,    # DCO
    'aBC': 9     # DCO
}

labeo_analysis = ThreePointCross(labeo_data)
labeo_analysis.report()

## Part 6: Visualizing Interference

Let's create visual representations of how interference affects genetic mapping.

In [ ]:
def plot_interference_comparison():
    """
    Compare expected vs observed DCO frequencies under different interference levels
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Parameters
    rf1, rf2 = 0.20, 0.15
    total = 1000
    expected_dco = rf1 * rf2 * total
    
    interference_levels = [
        (0.0, "No Interference (I = 0)"),
        (0.5, "Moderate Interference (I = 0.5)"),
        (1.0, "Complete Interference (I = 1.0)")
    ]
    
    for idx, (interference, title) in enumerate(interference_levels):
        ax = axes[idx]
        
        # Calculate observed based on interference
        coc = 1 - interference
        observed_dco = expected_dco * coc
        
        # Calculate other classes (simplified)
        sco1 = (rf1 * total) - observed_dco
        sco2 = (rf2 * total) - observed_dco
        parental = total - sco1 - sco2 - (2 * observed_dco)
        
        # Create bar chart
        categories = ['Parental', 'SCO\nRegion I', 'SCO\nRegion II', 'DCO']
        counts = [parental, sco1, sco2, observed_dco * 2]  # DCO × 2 for both classes
        colors = ['lightblue', 'lightcoral', 'lightgreen', 'gold']
        
        bars = ax.bar(categories, counts, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
        
        # Add count labels
        for bar, count in zip(bars, counts):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{count:.0f}',
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
        
        # Add expected DCO line
        if interference > 0:
            ax.axhline(y=expected_dco * 2, color='red', linestyle='--', 
                      linewidth=2, label=f'Expected DCO = {expected_dco*2:.0f}')
            ax.legend()
        
        ax.set_ylabel('Number of Offspring', fontsize=12)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_ylim(0, 700)
        ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/interference_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_interference_comparison()
print("\n✓ Interference comparison visualization created!")

In [ ]:
def plot_coc_vs_distance():
    """
    Show how COC typically varies with map distance
    (Generally, interference decreases as distance increases)
    """
    distances = np.linspace(5, 50, 50)  # cM
    
    # Typical patterns for different organisms
    # Strong interference organism (e.g., Drosophila)
    coc_strong = 0.1 + 0.9 * (1 - np.exp(-distances / 20))
    interference_strong = 1 - coc_strong
    
    # Moderate interference (e.g., fish, maize)
    coc_moderate = 0.3 + 0.7 * (1 - np.exp(-distances / 15))
    interference_moderate = 1 - coc_moderate
    
    # Weak interference (e.g., yeast)
    coc_weak = 0.6 + 0.4 * (1 - np.exp(-distances / 10))
    interference_weak = 1 - coc_weak
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: COC vs Distance
    ax1.plot(distances, coc_strong, 'b-', linewidth=3, label='Strong interference\n(e.g., Drosophila)', marker='o', markersize=4)
    ax1.plot(distances, coc_moderate, 'g-', linewidth=3, label='Moderate interference\n(e.g., Fish, Maize)', marker='s', markersize=4)
    ax1.plot(distances, coc_weak, 'r-', linewidth=3, label='Weak interference\n(e.g., Yeast)', marker='^', markersize=4)
    ax1.axhline(y=1.0, color='gray', linestyle='--', linewidth=2, alpha=0.5, label='No interference (COC = 1)')
    
    ax1.set_xlabel('Map Distance Between Regions (cM)', fontsize=13)
    ax1.set_ylabel('Coefficient of Coincidence (COC)', fontsize=13)
    ax1.set_title('COC Increases With Distance\n(Interference effect weakens)', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10, loc='lower right')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.1)
    
    # Plot 2: Interference vs Distance
    ax2.plot(distances, interference_strong, 'b-', linewidth=3, label='Strong interference', marker='o', markersize=4)
    ax2.plot(distances, interference_moderate, 'g-', linewidth=3, label='Moderate interference', marker='s', markersize=4)
    ax2.plot(distances, interference_weak, 'r-', linewidth=3, label='Weak interference', marker='^', markersize=4)
    ax2.axhline(y=0.0, color='gray', linestyle='--', linewidth=2, alpha=0.5, label='No interference (I = 0)')
    
    ax2.set_xlabel('Map Distance Between Regions (cM)', fontsize=13)
    ax2.set_ylabel('Interference (I)', fontsize=13)
    ax2.set_title('Interference Decreases With Distance\n(Crossovers become independent)', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10, loc='upper right')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(-0.1, 1.0)
    
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/coc_vs_distance.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_coc_vs_distance()
print("\n✓ COC vs distance relationship visualized!")
print("\nKey insight: As regions get farther apart, interference weakens.")
print("At large distances, crossovers become essentially independent (COC → 1, I → 0).")

## Part 7: Mapping Functions That Account for Interference

### Haldane vs Kosambi Functions

**Haldane's Function** (assumes NO interference, COC = 1):
$$d = -\frac{1}{2} \ln(1 - 2r)$$

**Kosambi's Function** (accounts for interference):
$$d = \frac{1}{4} \ln\left(\frac{1 + 2r}{1 - 2r}\right)$$

Kosambi function assumes moderate interference and is more accurate for most organisms.

In [ ]:
def haldane_function(r):
    """Convert RF to map distance (no interference)"""
    return -0.5 * np.log(1 - 2 * r)

def kosambi_function(r):
    """Convert RF to map distance (with interference)"""
    return 0.25 * np.log((1 + 2 * r) / (1 - 2 * r))

def compare_mapping_functions():
    """
    Compare Haldane and Kosambi mapping functions
    """
    r_values = np.linspace(0.01, 0.49, 100)
    
    # Calculate map distances
    haldane_distances = [haldane_function(r) * 100 for r in r_values]  # Convert to cM
    kosambi_distances = [kosambi_function(r) * 100 for r in r_values]
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Both functions
    ax1.plot(r_values * 100, haldane_distances, 'b-', linewidth=3, 
            label='Haldane (no interference)', marker='o', markersize=3, markevery=10)
    ax1.plot(r_values * 100, kosambi_distances, 'r-', linewidth=3, 
            label='Kosambi (with interference)', marker='s', markersize=3, markevery=10)
    ax1.plot([0, 50], [0, 50], 'k--', linewidth=2, alpha=0.3, label='1:1 line (r = d)')
    
    ax1.set_xlabel('Recombination Frequency (%)', fontsize=13)
    ax1.set_ylabel('Map Distance (cM)', fontsize=13)
    ax1.set_title('Mapping Functions: Haldane vs Kosambi', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 50)
    ax1.set_ylim(0, 120)
    
    # Plot 2: Difference between functions
    difference = np.array(haldane_distances) - np.array(kosambi_distances)
    ax2.plot(r_values * 100, difference, 'purple', linewidth=3, marker='o', markersize=3, markevery=10)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax2.fill_between(r_values * 100, 0, difference, alpha=0.3, color='purple')
    
    ax2.set_xlabel('Recombination Frequency (%)', fontsize=13)
    ax2.set_ylabel('Difference: Haldane - Kosambi (cM)', fontsize=13)
    ax2.set_title('Impact of Ignoring Interference\n(Haldane overestimates at high RF)', 
                 fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 50)
    
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/mapping_functions_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

compare_mapping_functions()

print("\n" + "="*70)
print("WHEN TO USE WHICH MAPPING FUNCTION")
print("="*70)
print("\n✓ Use HALDANE when:")
print("  - Interference is known to be very low (COC ≈ 1)")
print("  - Working with organisms like yeast or Neurospora")
print("  - Rough estimates are sufficient")
print("\n✓ Use KOSAMBI when:")
print("  - Most eukaryotic organisms (fish, plants, animals)")
print("  - Moderate interference expected")
print("  - More accurate mapping needed")
print("\n✓ Difference matters most when:")
print("  - RF > 20%")
print("  - High-resolution mapping")
print("  - QTL mapping and marker-assisted selection")
print("="*70)

## Part 8: Real-World Example - Earthworm Genomics

### Application to Mining Region Studies

Let's analyze a hypothetical three-point cross from earthworm populations in Western Odisha mining regions.

In [ ]:
print("\n\n" + "#"*70)
print("# EXAMPLE 2: Earthworm (Metaphire posthuma) Heavy Metal Tolerance")
print("# Three-Point Cross: Mining-Adapted vs Control Population")
print("#"*70)

print("\nHypothetical scenario:")
print("Markers linked to heavy metal tolerance genes in earthworms")
print("collected from Talcher coalfield region.\n")

earthworm_data = {
    'ABC': 245,  # Parental - mining adapted
    'abc': 238,  # Parental - control
    'AbC': 64,   # SCO region I
    'aBc': 69,   # SCO region I
    'ABc': 48,   # SCO region II  
    'abC': 52,   # SCO region II
    'Abc': 5,    # DCO
    'aBC': 4     # DCO
}

earthworm_analysis = ThreePointCross(earthworm_data)
earthworm_analysis.report()

# Additional visualization
fig, ax = plt.subplots(figsize=(14, 6))

categories = list(earthworm_data.keys())
counts = list(earthworm_data.values())

# Color code by class
colors = ['lightblue', 'lightblue',  # Parental
         'lightcoral', 'lightcoral',  # SCO I
         'lightgreen', 'lightgreen',  # SCO II
         'gold', 'gold']  # DCO

bars = ax.bar(categories, counts, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add labels
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{count}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add class labels
ax.text(0.5, -30, 'Parental', ha='center', fontsize=12, fontweight='bold', color='blue')
ax.text(2.5, -30, 'SCO I', ha='center', fontsize=12, fontweight='bold', color='red')
ax.text(4.5, -30, 'SCO II', ha='center', fontsize=12, fontweight='bold', color='green')
ax.text(6.5, -30, 'DCO', ha='center', fontsize=12, fontweight='bold', color='orange')

ax.set_xlabel('Gamete Class', fontsize=13)
ax.set_ylabel('Number of Offspring', fontsize=13)
ax.set_title('Earthworm Three-Point Cross Data\nHeavy Metal Tolerance Markers (Talcher Region)', 
            fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(-40, 300)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/earthworm_three_point_data.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("BIOLOGICAL SIGNIFICANCE FOR MINING REGION RESEARCH")
print("="*70)
print("\n✓ Tight linkage of markers suggests these tolerance genes")
print("  are clustered on the same chromosome")
print("\n✓ Moderate interference indicates normal meiotic behavior")
print("  even under heavy metal stress")
print("\n✓ Can use these markers for:")
print("  - Biomonitoring programs")
print("  - Population genetic surveys")
print("  - Identifying adaptive alleles in contaminated sites")
print("="*70)

## Part 9: Practice Problems

### Problem Set: Calculate COC and Interference

In [ ]:
def generate_practice_problem(problem_num, description, data):
    """
    Generate and solve a practice problem
    """
    print(f"\n{'='*70}")
    print(f"PRACTICE PROBLEM {problem_num}")
    print("="*70)
    print(f"\n{description}")
    print("\nData:")
    for key, value in data.items():
        print(f"  {key}: {value}")
    
    print("\nQuestions:")
    print("1. Calculate the recombination frequency for each region")
    print("2. Determine expected number of DCO")
    print("3. Calculate COC")
    print("4. Calculate Interference")
    print("5. Interpret the interference level")
    print("\n(Run next cell for solution)")
    
    return data

# Problem 1
problem1_data = {
    'Total offspring': 800,
    'SCO Region I': 144,
    'SCO Region II': 96,
    'DCO observed': 8
}

p1 = generate_practice_problem(
    1,
    "A three-point testcross in tomato plants.",
    problem1_data
)

In [ ]:
# Solution to Problem 1
print("\n" + "="*70)
print("SOLUTION TO PROBLEM 1")
print("="*70)

total = problem1_data['Total offspring']
sco1 = problem1_data['SCO Region I']
sco2 = problem1_data['SCO Region II']
dco_obs = problem1_data['DCO observed']

# Step 1: RF for each region
# Remember: recombinants = SCO + DCO
rf1 = (sco1 + dco_obs) / total
rf2 = (sco2 + dco_obs) / total

print("\n1. Recombination Frequencies:")
print(f"   Region I: ({sco1} + {dco_obs}) / {total} = {rf1:.4f} = {rf1*100:.2f}%")
print(f"   Region II: ({sco2} + {dco_obs}) / {total} = {rf2:.4f} = {rf2*100:.2f}%")

# Step 2: Expected DCO
expected_dco = rf1 * rf2 * total
print(f"\n2. Expected DCO:")
print(f"   {rf1:.4f} × {rf2:.4f} × {total} = {expected_dco:.2f}")

# Step 3: COC
coc = dco_obs / expected_dco
print(f"\n3. Coefficient of Coincidence:")
print(f"   COC = {dco_obs} / {expected_dco:.2f} = {coc:.3f}")

# Step 4: Interference
interference = 1 - coc
print(f"\n4. Interference:")
print(f"   I = 1 - {coc:.3f} = {interference:.3f} ({interference*100:.1f}%)")

# Step 5: Interpretation
print(f"\n5. Interpretation:")
if interference < 0.3:
    level = "WEAK"
elif interference < 0.6:
    level = "MODERATE"
else:
    level = "STRONG"

print(f"   This shows {level} interference ({interference*100:.1f}%).")
print(f"   One crossover reduces the probability of a nearby")
print(f"   second crossover by {interference*100:.1f}%.")
print(f"\n   {expected_dco - dco_obs:.1f} fewer DCOs than expected under independence.")
print("="*70)

## Part 10: Summary and Key Takeaways

### What We've Learned

1. **Interference is Real**: Crossovers are NOT truly independent - one crossover inhibits nearby ones

2. **COC Quantifies the Effect**: 
   - COC = Observed DCO / Expected DCO
   - COC < 1 means positive interference (typical)
   - COC = 1 means no interference

3. **Interference = 1 - COC**:
   - Tells us % reduction in double crossovers
   - Varies by organism and distance

4. **Practical Impact**:
   - Haldane function (no interference) overestimates map distances
   - Kosambi function (with interference) more accurate
   - Critical for accurate genetic maps

5. **Pattern Hunter Insight**:
   - "Chromosome uses interference to control recombination"
   - "Biological constraint on randomness"
   - "Mathematical model (Poisson) needs biological correction (interference)"

### Applications

- **QTL Mapping**: More accurate marker positions
- **Breeding Programs**: Better prediction of recombinant frequencies
- **Conservation Genetics**: Understanding gene flow patterns
- **Environmental Genomics**: Mapping adaptation alleles

### For Your Research

**Labeo rohita**:
- Use Kosambi function for microsatellite mapping
- Expect moderate interference (I ≈ 0.3-0.5)

**Earthworm genomics**:
- Interference data scarce - your research can fill this gap!
- Important for mapping heavy metal tolerance QTLs
- Could vary between mining-adapted vs control populations

## Formulas Quick Reference

### COC and Interference
```
Expected DCO = (RF₁) × (RF₂) × (Total offspring)

COC = Observed DCO / Expected DCO

Interference (I) = 1 - COC
```

### Recombination Frequencies
```
RF (Region I) = (SCO₁ + DCO) / Total

RF (Region II) = (SCO₂ + DCO) / Total
```

### Mapping Functions
```
Haldane:  d = -½ ln(1 - 2r)

Kosambi:  d = ¼ ln[(1 + 2r)/(1 - 2r)]
```

---

**Created for Pattern Hunters Educational Series**  
Dr. Ms Susama Kar and Alok Patel, Department of Zoology, Kuchinda College

This module extends our understanding of genetic mapping by showing how biological reality (interference) modifies mathematical predictions (Poisson distribution).